# 🤟 Train model VSL (Ngôn ngữ ký hiệu Việt Nam)

Notebook này chạy **miễn phí trên Google Colab**. Không cần cài gì trên máy.

**Cách dùng:** bấm nút ▶️ chạy **từng ô từ trên xuống**.

1. Cài thư viện
2. Tải dataset + **xem cấu trúc** (gửi kết quả ô này cho người hỗ trợ trước khi chạy tiếp)
3. Trích landmark từ video
4. Train model
5. Xuất model + tải về máy

> Mẹo: vào **Runtime → Change runtime type → T4 GPU** cho nhanh (không bắt buộc).


## 1️⃣ Cài thư viện (chạy 1 lần, ~2-3 phút)


In [ ]:
!pip -q install "mediapipe==0.10.14" "tensorflow==2.15.1" "tensorflowjs==4.17.0" \
  "huggingface_hub==0.24.6" "protobuf==4.25.3" opencv-python-headless tqdm scikit-learn
print('\n✅ Cài xong. NẾU ô sau báo lỗi import -> Runtime > Restart session, rồi chạy lại từ ô 2.')


## 2️⃣ Tải dataset & xem cấu trúc

⚠️ **Sau khi chạy ô này, copy phần output gửi cho người hỗ trợ** để xác nhận cách lấy nhãn trước khi train.


In [ ]:
import os, glob, collections
from huggingface_hub import snapshot_download

DATASET_REPO = 'star092304/ViSignLanguage-Video'  # đổi nếu repo khác
DATA_DIR = '/content/vsl_data'

path = snapshot_download(repo_id=DATASET_REPO, repo_type='dataset',
                         local_dir=DATA_DIR, local_dir_use_symlinks=False)
print('Đã tải về:', path)

VIDEO_EXTS = ('.mp4','.mov','.avi','.mkv','.webm')
videos = [p for p in glob.glob(os.path.join(DATA_DIR,'**','*'), recursive=True)
          if p.lower().endswith(VIDEO_EXTS)]
print(f'\nTổng số video: {len(videos)}')

print('\n--- 15 đường dẫn video mẫu (xem cấu trúc thư mục) ---')
for p in videos[:15]:
    print(p.replace(DATA_DIR+'/',''))

# nhãn đoán theo tên thư mục cha
labels = collections.Counter(os.path.basename(os.path.dirname(p)) for p in videos)
print(f'\n--- Số nhãn (theo thư mục cha): {len(labels)} ---')
for name, n in list(labels.items())[:20]:
    print(f'  {name}: {n} video')

print('\n--- File không phải video (metadata?) ---')
others = [p for p in glob.glob(os.path.join(DATA_DIR,'**','*'), recursive=True)
          if os.path.isfile(p) and not p.lower().endswith(VIDEO_EXTS)]
for p in others[:20]:
    print(p.replace(DATA_DIR+'/',''))


## 3️⃣ Trích landmark từ video

Lấy ~30 khung hình đều nhau mỗi video, dùng MediaPipe lấy toạ độ bàn tay.
Lưu tiến độ định kỳ để không mất nếu Colab ngắt.


In [ ]:
import numpy as np, cv2, json
import mediapipe as mp
from tqdm.auto import tqdm

# ===== Hằng số (PHẢI khớp app: vsl-web/lib/constants.ts) =====
NUM_HANDS=2; NUM_LANDMARKS=21; COORDS=3
FEATURES_PER_HAND=NUM_LANDMARKS*COORDS      # 63
FEATURES_PER_FRAME=NUM_HANDS*FEATURES_PER_HAND  # 126
SEQ_LEN=30; WRIST=0; MIDDLE_MCP=9
SAMPLE_FRAMES=45   # số khung lấy mẫu mỗi video (nhanh hơn đọc toàn bộ)

def normalize_hand(points):
    pts = points.astype(np.float32).copy()
    wrist = pts[WRIST].copy(); pts -= wrist
    scale = np.linalg.norm(pts[MIDDLE_MCP]) or 1.0
    pts /= scale
    return pts.reshape(-1)

def build_frame(hands):
    frame = np.zeros(FEATURES_PER_FRAME, dtype=np.float32)
    slot = {'Left':0,'Right':1}
    for h in hands:
        i = slot.get(h['label'])
        if i is None: continue
        frame[i*FEATURES_PER_HAND:(i+1)*FEATURES_PER_HAND] = normalize_hand(h['points'])
    return frame

def resample(frames, n):
    T=len(frames)
    if T==0: return np.zeros((n,FEATURES_PER_FRAME),dtype=np.float32)
    if T==n: return np.array(frames,dtype=np.float32)
    idx=np.linspace(0,T-1,n).round().astype(int)
    return np.array(frames,dtype=np.float32)[idx]

def sampled_frame_indices(total, k):
    if total<=0: return []
    if total<=k: return list(range(total))
    return list(np.linspace(0,total-1,k).round().astype(int))

def extract(path, hands):
    cap=cv2.VideoCapture(path)
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    want=set(sampled_frame_indices(total, SAMPLE_FRAMES)) if total else None
    frames=[]; i=0
    while True:
        ok,fr=cap.read()
        if not ok: break
        if want is None or i in want:
            res=hands.process(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB))
            hl=[]
            if res.multi_hand_landmarks and res.multi_handedness:
                for lm,hd in zip(res.multi_hand_landmarks,res.multi_handedness):
                    hl.append({'label':hd.classification[0].label,
                               'points':np.array([[p.x,p.y,p.z] for p in lm.landmark],dtype=np.float32)})
            if hl: frames.append(build_frame(hl))
        i+=1
    cap.release()
    return frames

# nhãn = tên thư mục cha (đổi ở đây nếu cấu trúc khác)
def label_of(p): return os.path.basename(os.path.dirname(p))

labels_sorted = sorted({label_of(p) for p in videos})
label_to_idx = {l:i for i,l in enumerate(labels_sorted)}
print(f'{len(videos)} video, {len(labels_sorted)} nhãn')

X=[]; y=[]
with mp.solutions.hands.Hands(static_image_mode=False, max_num_hands=NUM_HANDS,
                              min_detection_confidence=0.5, min_tracking_confidence=0.5) as hands:
    for n,p in enumerate(tqdm(videos, desc='Trích landmark')):
        fr=extract(p,hands)
        if not fr: continue
        X.append(resample(fr,SEQ_LEN)); y.append(label_to_idx[label_of(p)])
        if (n+1)%300==0:
            np.save('/content/X.npy',np.array(X,dtype=np.float32))
            np.save('/content/y.npy',np.array(y,dtype=np.int64))

X=np.array(X,dtype=np.float32); y=np.array(y,dtype=np.int64)
np.save('/content/X.npy',X); np.save('/content/y.npy',y)
json.dump(labels_sorted, open('/content/labels.json','w'), ensure_ascii=False, indent=2)
print(f'\n✅ Xong. X={X.shape}, y={y.shape}, nhãn={len(labels_sorted)}')


## 4️⃣ Train model (LSTM nhẹ, vài phút)


In [ ]:
import numpy as np, json, tensorflow as tf
from sklearn.model_selection import train_test_split

X=np.load('/content/X.npy'); y=np.load('/content/y.npy')
labels=json.load(open('/content/labels.json'))
num_classes=len(labels)
print('X',X.shape,'classes',num_classes)

Xtr,Xval,ytr,yval=train_test_split(X,y,test_size=0.15,random_state=42,stratify=y)

model=tf.keras.Sequential([
    tf.keras.layers.Input((SEQ_LEN,FEATURES_PER_FRAME)),
    tf.keras.layers.Masking(0.0),
    tf.keras.layers.LSTM(128,return_sequences=True),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64,activation='relu'),
    tf.keras.layers.Dense(num_classes,activation='softmax'),
])
model.compile('adam','sparse_categorical_crossentropy',metrics=['accuracy'])
cbs=[tf.keras.callbacks.EarlyStopping(patience=12,restore_best_weights=True,monitor='val_accuracy'),
     tf.keras.callbacks.ReduceLROnPlateau(patience=5,factor=0.5)]
model.fit(Xtr,ytr,validation_data=(Xval,yval),epochs=120,batch_size=32,callbacks=cbs)
print('\nVal accuracy:', round(float(model.evaluate(Xval,yval,verbose=0)[1]),3))
model.save('/content/vsl.keras')


## 5️⃣ Xuất model TF.js + tải về máy

Chạy xong sẽ tự tải file **`vsl-model.zip`**. Giải nén rồi làm theo hướng dẫn để đưa vào app.


In [ ]:
import tensorflowjs as tfjs, tensorflow as tf, shutil, os
m=tf.keras.models.load_model('/content/vsl.keras')
os.makedirs('/content/out/models/vsl',exist_ok=True)
tfjs.converters.save_keras_model(m,'/content/out/models/vsl')
shutil.copy('/content/labels.json','/content/out/labels.json')
shutil.make_archive('/content/vsl-model','zip','/content/out')
print('✅ Đã tạo /content/vsl-model.zip')
from google.colab import files
files.download('/content/vsl-model.zip')
